# AIMA Skin Lesion Segmentation - fixed 512x512 submission v5

This notebook is a submission-only correction for the completed controlled run.

Root cause of the low leaderboard score:
- the previous repaired submission restored masks to each original multi-megapixel test-image shape;
- the historical competition notebook resized every image to 512x512 and encoded the 512x512 prediction directly;
- therefore the competition submission contract is a fixed 512x512 mask, not original-image geometry.

This notebook does not retrain. It reloads the existing checkpoint, predicts at the repaired model's 256x256 resolution, applies the validation-selected post-processing in model space, resizes the final binary mask to 512x512 with nearest-neighbour interpolation, and then performs C-order RLE encoding.

Attach:
1. the repaired repository Dataset,
2. the original skin-lesion competition Dataset, and
3. the completed controlled-run artifact ZIP or Dataset.


## 1. User settings

In [ ]:
from pathlib import Path

# Submission-only correction. Do not change this to train.
WORKFLOW_MODE = "submit"

# Optional hints. Automatic discovery is used when these are None or invalid.
REPO_SOURCE_HINT = Path(
    "/kaggle/input/datasets/quyminhthangnguyen/"
    "aima-skin-lesion-segmentation-repaired/"
    "aima-skin-lesion-segmentation"
)
ARTIFACT_SOURCE_HINT = None
SAMPLE_SUBMISSION_PATH = None

# Historical competition dataset layout.
DATASET_ROOT = Path("/kaggle/input/warm-up-program-ai-vietnam-skin-segmentation")
TRAIN_IMAGE_DIR = DATASET_ROOT / "Train/Train/Image"
MASK_DIR = DATASET_ROOT / "Train/Train/Mask"
TEST_IMAGE_DIR = DATASET_ROOT / "Test/Test/Image"
GROUP_MAPPING_PATH = None

# "kaggle": keep Kaggle's TensorFlow build and install only support packages.
# "strict": install requirements-dev.txt exactly; may require restarting the session.
# "none": install nothing.
DEPENDENCY_MODE = "kaggle"

# Controlled-run settings. These match the completed run.
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 256

# Confirmed historical competition output geometry.
SUBMISSION_HEIGHT = 512
SUBMISSION_WIDTH = 512
BATCH_SIZE = 8
EPOCHS = 50
BASE_FILTERS = 32
EARLY_STOPPING_PATIENCE = 8
VALIDATION_FRACTION = 0.20
SEED = 42
LEARNING_RATE = 1e-3
L2_COEFFICIENT = 1e-4
MIXED_PRECISION = True
N_TTA = 8

THRESHOLD_GRID = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
MIN_COMPONENT_SIZES = [0, 16, 32, 64]
MORPHOLOGY_KERNELS = [0, 3]
MASK_SUFFIX = "_segmentation"

# Confirmed from the original competition notebook's actual encoder:
# np.ndarray.flatten() defaults to C-order.
RLE_ORDER = "C"
RLE_ORDER_CONFIRMED = True

# One controlled ablation only:
# use the same checkpoint and TTA, but no validation-selected post-processing.
SUBMISSION_VARIANT = "raw_threshold_0_5"
RAW_THRESHOLD = 0.50
RAW_MIN_COMPONENT_SIZE = 0
RAW_MORPHOLOGY_KERNEL = 0

WORK_ROOT = Path("/kaggle/working")
OUTPUT_DIR = WORK_ROOT / "artifacts" / "controlled_rerun"
CONFIG_PATH = WORK_ROOT / "kaggle_controlled_rerun.json"

if WORKFLOW_MODE not in {"verify", "train", "submit"}:
    raise ValueError("WORKFLOW_MODE must be 'verify', 'train', or 'submit'")

RUN_REPOSITORY_TESTS = WORKFLOW_MODE in {"verify", "train"}
RUN_PREPARE = WORKFLOW_MODE in {"verify", "train"}
RUN_SMOKE_CHECK = WORKFLOW_MODE in {"verify", "train"}
RUN_TRAINING = WORKFLOW_MODE == "train"
RUN_SUBMISSION = WORKFLOW_MODE == "submit"

print("Workflow mode:", WORKFLOW_MODE)
print("Training enabled:", RUN_TRAINING)
print("Submission enabled:", RUN_SUBMISSION)
print("Output directory:", OUTPUT_DIR)


## 2. Inspect runtime and attached inputs

In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())
print("Kaggle input exists:", Path("/kaggle/input").is_dir())

for path in sorted(Path("/kaggle/input").iterdir()):
    print(" -", path)

## 3. Locate and copy the repaired repository

In [ ]:
import shutil
import zipfile

REPO_ARCHIVE_NAMES = {
    "aima-skin-lesion-segmentation-repaired-v2.zip",
    "aima-skin-lesion-segmentation-repaired.zip",
}
repo_target = WORK_ROOT / "aima-skin-lesion-segmentation"


def is_project_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "pyproject.toml").is_file()
        and (path / "src" / "skin_lesion_segmentation").is_dir()
    )


def find_project_roots(root: Path) -> list[Path]:
    roots = []
    if is_project_root(root):
        roots.append(root)
    if root.is_dir():
        for pyproject in root.rglob("pyproject.toml"):
            candidate = pyproject.parent
            if is_project_root(candidate):
                roots.append(candidate)
    return sorted(set(roots))


def resolve_repository_source() -> tuple[str, Path]:
    if REPO_SOURCE_HINT is not None:
        hint = Path(REPO_SOURCE_HINT)
        if is_project_root(hint):
            return "directory", hint
        if hint.is_file() and hint.suffix.lower() == ".zip":
            return "zip", hint

    directory_candidates = []
    for pyproject in Path("/kaggle/input").rglob("pyproject.toml"):
        candidate = pyproject.parent
        if is_project_root(candidate):
            directory_candidates.append(candidate)
    directory_candidates = sorted(set(directory_candidates))
    if len(directory_candidates) == 1:
        return "directory", directory_candidates[0]

    zip_candidates = sorted(
        path for path in Path("/kaggle/input").rglob("*.zip")
        if path.name in REPO_ARCHIVE_NAMES
    )
    if len(zip_candidates) == 1:
        return "zip", zip_candidates[0]

    raise RuntimeError(
        "Could not identify exactly one repaired repository. "
        f"Directory candidates={directory_candidates}; ZIP candidates={zip_candidates}. "
        "Set REPO_SOURCE_HINT to the exact project directory or ZIP."
    )


source_kind, repo_source = resolve_repository_source()
print("Repository source:", repo_source)

if repo_target.exists():
    shutil.rmtree(repo_target)

if source_kind == "directory":
    shutil.copytree(repo_source, repo_target)
else:
    extraction_root = WORK_ROOT / "_repository_extracted"
    if extraction_root.exists():
        shutil.rmtree(extraction_root)
    extraction_root.mkdir(parents=True)
    with zipfile.ZipFile(repo_source) as archive:
        archive.extractall(extraction_root)
    project_roots = find_project_roots(extraction_root)
    if len(project_roots) != 1:
        raise RuntimeError(f"Expected one project root in archive, found {project_roots}")
    shutil.copytree(project_roots[0], repo_target)
    shutil.rmtree(extraction_root)

REPO_ROOT = repo_target
assert is_project_root(REPO_ROOT), REPO_ROOT
print("Repository copied to:", REPO_ROOT)

## 4. Compatibility repair for the older checkpoint test

In [ ]:
test_file = REPO_ROOT / "tests" / "test_tensorflow.py"
if not test_file.is_file():
    raise FileNotFoundError(test_file)

text = test_file.read_text(encoding="utf-8")
fixed_marker = "model.train_on_batch(x, y)"

if fixed_marker in text:
    print("Checkpoint round-trip test already uses built optimizer state.")
else:
    old = "\n".join([
        "    x = np.random.default_rng(1).random((1, 32, 32, 3), dtype=np.float32)",
        "    before = model.predict(x, verbose=0)",
        '    path = tmp_path / "model.keras"',
        "    model.save(path)",
        "    restored = load_model(path, compile=True)",
        "",
    ])
    new = "\n".join([
        "    x = np.random.default_rng(1).random((1, 32, 32, 3), dtype=np.float32)",
        "    y = np.zeros((1, 32, 32, 1), dtype=np.float32)",
        "    model.train_on_batch(x, y)",
        "    before = model.predict(x, verbose=0)",
        '    path = tmp_path / "model.keras"',
        "    model.save(path)",
        "    restored = load_model(path, compile=True)",
        "",
    ])
    if old not in text:
        raise RuntimeError(
            "The attached repository contains an unknown checkpoint-test version. "
            "Use the repaired-v2 repository Dataset."
        )
    test_file.write_text(text.replace(old, new), encoding="utf-8")
    print("Applied checkpoint-test compatibility repair.")

## 5. Install dependencies and project

In [ ]:
def run_command(command, *, cwd=REPO_ROOT):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    completed = subprocess.run(command, cwd=cwd, text=True)
    if completed.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {completed.returncode}: {' '.join(command)}"
        )
    return completed


if DEPENDENCY_MODE == "kaggle":
    run_command([
        sys.executable, "-m", "pip", "install", "-q",
        "albumentations==1.4.24",
        "opencv-python-headless==4.10.0.84",
        "pytest==8.3.4",
    ])
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
elif DEPENDENCY_MODE == "strict":
    run_command([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dev.txt"])
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
elif DEPENDENCY_MODE == "none":
    print("Dependency installation skipped.")
else:
    raise ValueError("DEPENDENCY_MODE must be 'kaggle', 'strict', or 'none'")

# Editable installs commonly add a .pth file that is processed only when a
# Python interpreter starts. Papermill keeps this kernel alive, so expose the
# src-layout package explicitly and verify the import in this process.
import importlib

SOURCE_ROOT = (REPO_ROOT / "src").resolve()
if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(SOURCE_ROOT)
source_root_text = str(SOURCE_ROOT)
if source_root_text not in sys.path:
    sys.path.insert(0, source_root_text)
importlib.invalidate_caches()

package = importlib.import_module("skin_lesion_segmentation")
print("Package import verified:", Path(package.__file__).resolve())


## 6. Verify package and GPU visibility

In [ ]:
import importlib.metadata
import numpy as np
import pandas as pd
import tensorflow as tf

for package in [
    "tensorflow", "keras", "numpy", "opencv-python-headless",
    "albumentations", "pandas", "pytest"
]:
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: not installed")

print("GPUs:", tf.config.list_physical_devices("GPU"))
subprocess.run(["nvidia-smi"], check=False)

## 7. Run repository verification when requested

In [ ]:
if RUN_REPOSITORY_TESTS:
    run_command([sys.executable, "-m", "pytest", "-q", "-W", "error"])
    run_command([sys.executable, "scripts/verify_repository.py"])
    run_command([sys.executable, "scripts/synthetic_smoke.py"])
else:
    print("Repository tests skipped in submission mode.")

## 8. Validate dataset paths and resolve the official sample submission

In [ ]:
from skin_lesion_segmentation.data import SUPPORTED_IMAGE_EXTENSIONS
from skin_lesion_segmentation.inference import discover_test_images
from skin_lesion_segmentation.submission import infer_submission_id_suffix


def supported_files(directory: Path) -> list[Path]:
    directory = Path(directory)
    if not directory.is_dir():
        return []
    return sorted(
        path for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
    )


for label, directory in [
    ("Training images", TRAIN_IMAGE_DIR),
    ("Training masks", MASK_DIR),
    ("Test images", TEST_IMAGE_DIR),
]:
    files = supported_files(directory)
    print(f"{label}: {directory} ({len(files)} supported files)")
    if not directory.is_dir():
        raise FileNotFoundError(directory)

csv_candidates = sorted(Path("/kaggle/input").rglob("*.csv"))
print("CSV candidates:")
for path in csv_candidates:
    print(" -", path)


def resolve_sample_submission(configured_path=None) -> Path:
    if configured_path is not None:
        configured = Path(configured_path)
        if configured.is_file():
            return configured
        raise FileNotFoundError(configured)

    exact_names = {
        "sample_submission.csv",
        "sample-submission.csv",
        "samplesubmission.csv",
    }
    exact = [path for path in csv_candidates if path.name.casefold() in exact_names]
    if len(exact) == 1:
        return exact[0]

    named = [
        path for path in csv_candidates
        if "sample" in path.name.casefold() and "submission" in path.name.casefold()
    ]
    if len(named) == 1:
        return named[0]

    raise RuntimeError(
        "Could not identify exactly one official sample submission. "
        f"Exact candidates={exact}; named candidates={named}. "
        "Set SAMPLE_SUBMISSION_PATH in the settings cell."
    )


SAMPLE_SUBMISSION_PATH = resolve_sample_submission(SAMPLE_SUBMISSION_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH, keep_default_na=False)
print("Sample submission:", SAMPLE_SUBMISSION_PATH)
print("Columns:", sample_submission.columns.tolist())
print("Rows:", len(sample_submission))
display(sample_submission.head())

if len(sample_submission.columns) != 2:
    raise ValueError("Expected the official sample submission to contain exactly two columns")

SUBMISSION_ID_COLUMN = str(sample_submission.columns[0])
SUBMISSION_MASK_COLUMN = str(sample_submission.columns[1])
test_paths = discover_test_images(TEST_IMAGE_DIR)
SUBMISSION_ID_SUFFIX = infer_submission_id_suffix(
    test_paths,
    sample_submission,
    id_column=SUBMISSION_ID_COLUMN,
)

print("Submission ID column:", SUBMISSION_ID_COLUMN)
print("Submission mask column:", SUBMISSION_MASK_COLUMN)
print("Template-proven ID suffix:", repr(SUBMISSION_ID_SUFFIX))

## 9. Restore existing controlled-run artifacts

In `submit` mode this cell does not train. It first uses artifacts already present under `/kaggle/working`. If they are absent, it searches attached inputs for the artifact ZIP or a directory containing the required files.

In [ ]:
REQUIRED_SUBMISSION_ARTIFACTS = {
    "best_model.keras",
    "chosen_postprocessing.json",
}


def artifact_set_complete(directory: Path) -> bool:
    directory = Path(directory)
    return directory.is_dir() and all(
        (directory / name).is_file() for name in REQUIRED_SUBMISSION_ARTIFACTS
    )


def copy_artifact_directory(source: Path, destination: Path) -> None:
    source = Path(source)
    destination.mkdir(parents=True, exist_ok=True)
    for item in source.iterdir():
        target = destination / item.name
        if item.is_file():
            shutil.copy2(item, target)
        elif item.is_dir():
            if target.exists():
                shutil.rmtree(target)
            shutil.copytree(item, target)


def restore_run_artifacts(
    output_dir: Path,
    *,
    source_hint=None,
    input_root: Path = Path("/kaggle/input"),
) -> str:
    output_dir = Path(output_dir)
    if artifact_set_complete(output_dir):
        return f"existing working directory: {output_dir}"

    sources = []
    if source_hint is not None:
        hint = Path(source_hint)
        if hint.exists():
            sources.append(hint)
        else:
            raise FileNotFoundError(hint)

    if not sources:
        accepted_artifact_archives = {
            "aima_skin_lesion_controlled_rerun_artifacts.zip",
            "aima_skin_lesion_controlled_rerun_artifacts_with_submission.zip",
            "aima_skin_lesion_fixed_512_submission_artifacts.zip",
        }
        zip_candidates = sorted(
            path
            for path in input_root.rglob("*.zip")
            if path.name in accepted_artifact_archives
        )
        complete_directories = sorted({
            checkpoint.parent
            for checkpoint in input_root.rglob("best_model.keras")
            if artifact_set_complete(checkpoint.parent)
        })

        if len(zip_candidates) == 1:
            sources = [zip_candidates[0]]
        elif len(complete_directories) == 1:
            sources = [complete_directories[0]]
        elif len(zip_candidates) + len(complete_directories) == 0:
            raise RuntimeError(
                "No controlled-run artifacts were found. Attach the controlled-run "
                "artifact ZIP or a prior "
                "notebook output containing best_model.keras and "
                "chosen_postprocessing.json."
            )
        else:
            raise RuntimeError(
                "Multiple artifact sources were found. Set ARTIFACT_SOURCE_HINT "
                f"to one exact ZIP or directory. ZIPs={zip_candidates}; "
                f"directories={complete_directories}"
            )

    source = sources[0]
    output_dir.mkdir(parents=True, exist_ok=True)

    if source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as archive:
            archive.extractall(output_dir)
        description = f"artifact ZIP: {source}"
    elif source.is_dir():
        copy_artifact_directory(source, output_dir)
        description = f"artifact directory: {source}"
    else:
        raise ValueError(f"Unsupported artifact source: {source}")

    if not artifact_set_complete(output_dir):
        missing = sorted(
            name for name in REQUIRED_SUBMISSION_ARTIFACTS
            if not (output_dir / name).is_file()
        )
        raise RuntimeError(
            f"Artifact restoration from {source} did not produce required files: {missing}"
        )
    return description


if RUN_SUBMISSION:
    artifact_source_description = restore_run_artifacts(
        OUTPUT_DIR,
        source_hint=ARTIFACT_SOURCE_HINT,
    )
    print("Restored submission artifacts from", artifact_source_description)
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("Artifact restoration is only mandatory in submission mode.")

for path in sorted(OUTPUT_DIR.iterdir()):
    print(f" - {path.name}: {path.stat().st_size} bytes")


## 10. Write the effective configuration

In [ ]:
import json
from skin_lesion_segmentation.config import ExperimentConfig

config_data = {
    "image_dir": str(TRAIN_IMAGE_DIR),
    "mask_dir": str(MASK_DIR),
    "test_image_dir": str(TEST_IMAGE_DIR),
    "sample_submission_path": str(SAMPLE_SUBMISSION_PATH),
    "group_mapping_path": str(GROUP_MAPPING_PATH) if GROUP_MAPPING_PATH is not None else None,
    "output_dir": str(OUTPUT_DIR),
    "mask_suffix": MASK_SUFFIX,
    "image_height": IMAGE_HEIGHT,
    "image_width": IMAGE_WIDTH,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "base_filters": BASE_FILTERS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "validation_fraction": VALIDATION_FRACTION,
    "seed": SEED,
    "learning_rate": LEARNING_RATE,
    "l2_coefficient": L2_COEFFICIENT,
    "mixed_precision": MIXED_PRECISION,
    "rle_order": RLE_ORDER,
    "rle_order_confirmed": RLE_ORDER_CONFIRMED,
    "submission_id_column": SUBMISSION_ID_COLUMN,
    "submission_mask_column": SUBMISSION_MASK_COLUMN,
    "submission_id_suffix": SUBMISSION_ID_SUFFIX,
    "n_tta": N_TTA,
    "threshold_grid": THRESHOLD_GRID,
    "min_component_sizes": MIN_COMPONENT_SIZES,
    "morphology_kernels": MORPHOLOGY_KERNELS,
}

config = ExperimentConfig(**config_data)
config.validate()
config.save(CONFIG_PATH)
print(CONFIG_PATH.read_text())

## 11. Prepare pairing and split manifests when requested

In [ ]:
if RUN_PREPARE:
    run_command([
        sys.executable, "-m", "skin_lesion_segmentation.cli",
        "prepare", "--config", CONFIG_PATH,
    ])
    pair_manifest = pd.read_csv(OUTPUT_DIR / "pair_manifest.csv")
    split_manifest = pd.read_csv(OUTPUT_DIR / "split_manifest.csv")
    print("Pairs:", len(pair_manifest))
    print(split_manifest["split"].value_counts())
    display(split_manifest.head())
else:
    print("Preparation skipped in submission mode.")

## 12. Model forward-pass smoke check when requested

In [ ]:
if RUN_SMOKE_CHECK:
    from skin_lesion_segmentation.losses import combined_segmentation_loss
    from skin_lesion_segmentation.model import build_attention_unet

    smoke_model = build_attention_unet(
        (IMAGE_HEIGHT, IMAGE_WIDTH, 3),
        base_filters=BASE_FILTERS,
        l2_coefficient=L2_COEFFICIENT,
        mixed_precision=MIXED_PRECISION,
    )
    x = np.random.default_rng(SEED).random(
        (1, IMAGE_HEIGHT, IMAGE_WIDTH, 3), dtype=np.float32
    )
    y = np.zeros((1, IMAGE_HEIGHT, IMAGE_WIDTH, 1), dtype=np.float32)
    prediction = smoke_model(x, training=False)
    loss = combined_segmentation_loss(
        tf.convert_to_tensor(y),
        tf.cast(prediction, tf.float32),
    )
    print("Output shape:", prediction.shape)
    print("Output dtype:", prediction.dtype)
    print("Loss dtype:", loss.dtype)
    print("Loss value:", float(loss.numpy()))
    if prediction.dtype != tf.float32 or loss.dtype != tf.float32:
        raise TypeError("Model output and loss must be float32")
    if not np.isfinite(float(loss.numpy())):
        raise FloatingPointError("Smoke-test loss is not finite")
else:
    print("Model smoke check skipped in submission mode.")

## 13. Controlled training when requested

In [ ]:
if RUN_TRAINING:
    if not tf.config.list_physical_devices("GPU"):
        raise RuntimeError("Training mode requires a visible GPU")
    run_command([
        sys.executable, "-m", "skin_lesion_segmentation.cli",
        "train", "--config", CONFIG_PATH,
    ])
else:
    print("Training skipped.")

## 14. Inspect existing run evidence

In [ ]:
from IPython.display import Image, display as display_image
import matplotlib.pyplot as plt

metrics_path = OUTPUT_DIR / "final_validation_metrics.json"
history_path = OUTPUT_DIR / "training_history.csv"
qualitative_path = OUTPUT_DIR / "qualitative_validation.png"

if metrics_path.is_file():
    print("Validation metrics:")
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
else:
    print("No validation metrics JSON is present.")

if history_path.is_file():
    history = pd.read_csv(history_path)
    display(history.tail())
    if "loss" in history and "val_loss" in history:
        history[["loss", "val_loss"]].plot()
        plt.title("Training history")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.show()

if qualitative_path.is_file():
    display_image(Image(filename=str(qualitative_path)))

## 15. Verify checkpoint checksum

In [ ]:
checkpoint_path = OUTPUT_DIR / "best_model.keras"
checkpoint_record_path = OUTPUT_DIR / "checkpoint.json"

if checkpoint_path.is_file():
    print("Checkpoint:", checkpoint_path)
    run_command([sys.executable, "scripts/checkpoint_checksum.py", checkpoint_path])
else:
    print("No checkpoint found.")

if checkpoint_record_path.is_file():
    print(checkpoint_record_path.read_text())

## 16. Generate the corrected fixed-512 submission

This cell is restart-safe and reconstructs all required paths. It deliberately does not use the repository's previous original-image restoration submission path.

Processing order:
1. predict probabilities at 256x256 with eight-transform TTA;
2. threshold and post-process at 256x256 using the validation-selected parameters;
3. resize the final binary mask to exactly 512x512 with nearest-neighbour interpolation;
4. encode the 512x512 mask using confirmed C-order RLE;
5. validate IDs, columns, row count, RLE bounds, and template ordering.


In [ ]:
from pathlib import Path
import json
import shutil
import sys

import cv2
import numpy as np
import pandas as pd

from skin_lesion_segmentation.artifacts import RunArtifacts
from skin_lesion_segmentation.inference import discover_test_images
from skin_lesion_segmentation.loading import PathImageSequence
from skin_lesion_segmentation.model import load_model
from skin_lesion_segmentation.postprocessing import apply_postprocessing
from skin_lesion_segmentation.submission import build_submission
from skin_lesion_segmentation.tta import predict_with_tta

# Reconstruct all submission state inside this cell.
work_root = Path("/kaggle/working")
output_dir = work_root / "artifacts" / "controlled_rerun"
config_path = work_root / "kaggle_controlled_rerun.json"
checkpoint_path = output_dir / "best_model.keras"
postprocessing_path = output_dir / "chosen_postprocessing.json"
submission_path = output_dir / "submission_raw_threshold_0_5.csv"
submission_summary_path = output_dir / "submission_raw_threshold_0_5_summary.json"

if not RUN_SUBMISSION:
    raise RuntimeError("This notebook is submission-only; RUN_SUBMISSION must be True.")

restore_source = restore_run_artifacts(
    output_dir,
    source_hint=ARTIFACT_SOURCE_HINT,
)
print("Artifacts restored from:", restore_source)

required_paths = {
    "configuration": config_path,
    "sample submission": Path(SAMPLE_SUBMISSION_PATH),
    "checkpoint": checkpoint_path,
    "post-processing parameters": postprocessing_path,
}
missing = {
    label: path for label, path in required_paths.items()
    if not path.is_file()
}
if missing:
    raise FileNotFoundError(
        "Submission prerequisites are missing: "
        + ", ".join(f"{label}={path}" for label, path in missing.items())
    )

if RLE_ORDER.upper() != "C" or not RLE_ORDER_CONFIRMED:
    raise RuntimeError(
        "The historical successful notebook used np.ndarray.flatten(), "
        "which is C-order. Keep RLE_ORDER='C' and RLE_ORDER_CONFIRMED=True."
    )

validation_selected_params = json.loads(
    postprocessing_path.read_text(encoding="utf-8")
)
required_param_names = {"threshold", "min_component_size", "morphology_kernel"}
if set(validation_selected_params) != required_param_names:
    raise ValueError(
        f"Post-processing parameters must be exactly {sorted(required_param_names)}, "
        f"got {sorted(validation_selected_params)}"
    )

# Controlled raw-mask ablation. This is based on validation evidence:
# the selected post-processing gain was small and inconsistent, and it hurt
# smaller lesions. Do not change these values using leaderboard feedback.
params = {
    "threshold": float(RAW_THRESHOLD),
    "min_component_size": int(RAW_MIN_COMPONENT_SIZE),
    "morphology_kernel": int(RAW_MORPHOLOGY_KERNEL),
}
print("Validation-selected parameters:", validation_selected_params)
print("Applied submission parameters:", params)

model = load_model(checkpoint_path, compile=False)
test_paths = discover_test_images(TEST_IMAGE_DIR)
sequence = PathImageSequence(
    test_paths,
    (IMAGE_HEIGHT, IMAGE_WIDTH),
    BATCH_SIZE,
)

predictions = []
for batch_index in range(len(sequence)):
    images = sequence[batch_index]
    probabilities = predict_with_tta(
        lambda transformed: model.predict(transformed, verbose=0),
        images,
        n_augmentations=N_TTA,
    )
    for probability in probabilities[..., 0]:
        raw_mask = probability >= float(params["threshold"])
        model_space_mask = apply_postprocessing(
            raw_mask,
            int(params["min_component_size"]),
            int(params["morphology_kernel"]),
        ).astype(np.uint8)

        # Competition contract: fixed 512x512 mask, matching the historical
        # notebook that produced the credible leaderboard scores.
        submission_mask = cv2.resize(
            model_space_mask,
            (SUBMISSION_WIDTH, SUBMISSION_HEIGHT),
            interpolation=cv2.INTER_NEAREST,
        )
        submission_mask = (submission_mask > 0).astype(np.uint8)

        if submission_mask.shape != (SUBMISSION_HEIGHT, SUBMISSION_WIDTH):
            raise RuntimeError(
                f"Unexpected submission mask shape: {submission_mask.shape}"
            )
        if not set(np.unique(submission_mask)).issubset({0, 1}):
            raise RuntimeError("Submission mask is not binary")
        predictions.append(submission_mask)

if len(predictions) != len(test_paths):
    raise RuntimeError(
        f"Prediction count mismatch: {len(predictions)} predictions for "
        f"{len(test_paths)} test images"
    )

sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH, keep_default_na=False)
submission, summary = build_submission(
    test_paths,
    predictions,
    sample_submission=sample_submission,
    id_column=SUBMISSION_ID_COLUMN,
    mask_column=SUBMISSION_MASK_COLUMN,
    submission_id_suffix=SUBMISSION_ID_SUFFIX,
    rle_order=RLE_ORDER,
)

# Strong RLE geometry check: no run may exceed 512*512.
maximum_valid_position = SUBMISSION_HEIGHT * SUBMISSION_WIDTH
max_encoded_position = 0
malformed_rows = []

for row_index, encoded in enumerate(submission[SUBMISSION_MASK_COLUMN].astype(str)):
    encoded = encoded.strip()
    if not encoded:
        continue
    values = np.fromstring(encoded, dtype=np.int64, sep=" ")
    if len(values) % 2 != 0:
        malformed_rows.append(row_index)
        continue
    starts = values[0::2]
    lengths = values[1::2]
    ends = starts + lengths - 1
    if np.any(starts < 1) or np.any(lengths < 1):
        malformed_rows.append(row_index)
    if len(ends):
        max_encoded_position = max(max_encoded_position, int(ends.max()))

if malformed_rows:
    raise RuntimeError(f"Malformed RLE rows: {malformed_rows[:10]}")
if max_encoded_position > maximum_valid_position:
    raise RuntimeError(
        f"RLE exceeds fixed 512x512 bounds: max={max_encoded_position}, "
        f"allowed={maximum_valid_position}"
    )

output_dir.mkdir(parents=True, exist_ok=True)
submission.to_csv(submission_path, index=False)

result = {
    **summary,
    "path": str(submission_path),
    "checkpoint": str(checkpoint_path),
    "submission_variant": SUBMISSION_VARIANT,
    "validation_selected_postprocessing": validation_selected_params,
    "applied_postprocessing": params,
    "rle_order_confirmed": True,
    "submission_mask_height": SUBMISSION_HEIGHT,
    "submission_mask_width": SUBMISSION_WIDTH,
    "maximum_valid_rle_position": maximum_valid_position,
    "maximum_observed_rle_position": max_encoded_position,
    "geometry_contract": (
        "fixed 512x512, matching the historical competition notebook; "
        "not original source-image dimensions"
    ),
}
submission_summary_path.write_text(
    json.dumps(result, indent=2) + "\n",
    encoding="utf-8",
)

print(json.dumps(result, indent=2))
display(submission.head())
print("Submission written:", submission_path)
print("Summary written:", submission_summary_path)


## 17. Package outputs

In [ ]:
artifact_archive = WORK_ROOT / "aima_skin_lesion_raw_threshold_0_5_submission_artifacts.zip"

if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    if artifact_archive.exists():
        artifact_archive.unlink()
    shutil.make_archive(
        str(artifact_archive.with_suffix("")),
        "zip",
        root_dir=OUTPUT_DIR,
    )
    print("Packaged artifacts:", artifact_archive)
    print("Size:", artifact_archive.stat().st_size, "bytes")
else:
    print("No artifacts are available to package.")


## 18. Final actions

Confirm that section 16 reports:

- `sample_submission_validated: true`
- `rle_order: C`
- `submission_mask_height: 512`
- `submission_mask_width: 512`
- `maximum_observed_rle_position <= 262144`

Download:

- `/kaggle/working/artifacts/controlled_rerun/submission.csv`
- `/kaggle/working/artifacts/controlled_rerun/submission_validation_summary.json`
- `/kaggle/working/aima_skin_lesion_fixed_512_submission_artifacts.zip`

Submit the new CSV as a separate Kaggle submission. Do not delete the previous low-scoring submission; retain it as evidence of the geometry-contract bug.
